# Building a Linear Programming (LP) model from scratch

**Goal:** Implement LP modeling functions, solve a real supply chain problem, and perform scenario analysis.

## Problem Recap

We're optimizing milk sourcing across 3 plants over 12 months to minimize total cost.

### Decision Variables (252 total)

- **x[p, s, t]** — volume bought from spot supplier s at plant p in month t  
  (3 plants × 3 suppliers × 12 months = 108 variables)

- **u[r, t]** — volume transferred on route r in month t  
  (6 routes × 12 months = 72 variables)

- **e[p, k, t]** — excess milk sold from plant p to customer k in month t  
  (3 plants × 2 customers × 12 months = 72 variables)

### Constraints

- **36 balance constraints** — net inflow = production need at each (plant, month)
- **12 S3 capacity constraints** — total S3 purchases ≤ 150,000 L per month

### The Indexing Challenge

`scipy.linprog` expects a **flat 1D vector** of 252 variables. We need helper functions to map logical coordinates (plant, supplier, month) → integer index.

In [ ]:
# Setup
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from milk_balancing.constants import NK, NP, NS, ROUTES, T, MONTHS

print("Problem dimensions:")
print(f"  Plants: {NP}")
print(f"  Spot suppliers: {NS}")
print(f"  Customers: {NK}")
print(f"  Months: {T}")
print(f"  Transfer routes: {len(ROUTES)}")
print(f"  Routes: {ROUTES}")
print(f"\nTotal variables: {NP * NS * T + len(ROUTES) * T + NP * NK * T}")

---

## Part 1: Variable Indexing (20 min)

### Context

We organize the 252 variables in three consecutive blocks:

```
[ x[0,0,0] ... x[2,2,11] | u[route0,0] ... u[route5,11] | e[0,0,0] ... e[2,1,11] ]
  indices 0-107          | indices 108-179              | indices 180-251
```

### Your Task

Implement the three indexing functions below. Each maps logical coordinates to a flat index.

**Hint:** For `idx_x`, think about how to flatten a 3D array with shape (NP, NS, T) into 1D.

In [ ]:
# YOUR CODE HERE
def idx_x(p: int, s: int, t: int) -> int:
    """Flat index of spot-purchase variable x[p, s, t].  Range: 0 … 107."""
    raise NotImplementedError("Implement idx_x() function.")


def idx_u(r: tuple, t: int) -> int:
    """Flat index of transfer variable u[r, t].  Range: 108 … 179."""
    raise NotImplementedError("Implement idx_u() function.")


def idx_e(p: int, k: int, t: int) -> int:
    """Flat index of excess-sales variable e[p, k, t].  Range: 180 … 251."""
    raise NotImplementedError("Implement idx_e() function.")

### Test Your Implementation

In [ ]:
# Basic boundary tests
assert idx_x(0, 0, 0) == 0, "First x variable should be at index 0"
assert idx_x(2, 2, 11) == 107, "Last x variable should be at index 107"

assert idx_u(ROUTES[0], 0) == 108, "First u variable should be at index 108"
assert idx_u(ROUTES[-1], 11) == 179, "Last u variable should be at index 179"

assert idx_e(0, 0, 0) == 180, "First e variable should be at index 180"
assert idx_e(2, 1, 11) == 251, "Last e variable should be at index 251"

print("✅ Boundary tests passed!")

# Uniqueness test
all_indices = set()

for p in range(NP):
    for s in range(NS):
        for t in range(T):
            idx = idx_x(p, s, t)
            assert 0 <= idx <= 107, f"idx_x({p},{s},{t}) = {idx} out of range"
            assert idx not in all_indices, f"Duplicate index {idx}"
            all_indices.add(idx)

for r in ROUTES:
    for t in range(T):
        idx = idx_u(r, t)
        assert 108 <= idx <= 179, f"idx_u({r},{t}) = {idx} out of range"
        assert idx not in all_indices, f"Duplicate index {idx}"
        all_indices.add(idx)

for p in range(NP):
    for k in range(NK):
        for t in range(T):
            idx = idx_e(p, k, t)
            assert 180 <= idx <= 251, f"idx_e({p},{k},{t}) = {idx} out of range"
            assert idx not in all_indices, f"Duplicate index {idx}"
            all_indices.add(idx)

assert len(all_indices) == 252, f"Expected 252 unique indices, got {len(all_indices)}"
print("✅ All 252 indices are unique and within correct ranges!")

<details>
<summary><b>💡 Solution - Part 1</b> (click to expand — try on your own first!)</summary>

```python
def idx_x(p: int, s: int, t: int) -> int:
    """Flat index of spot-purchase variable x[p, s, t].  Range: 0 … 107."""
    return p * NS * T + s * T + t


def idx_u(r: tuple, t: int) -> int:
    """Flat index of transfer variable u[r, t].  Range: 108 … 179."""
    return NP * NS * T + ROUTES.index(r) * T + t


def idx_e(p: int, k: int, t: int) -> int:
    """Flat index of excess-sales variable e[p, k, t].  Range: 180 … 251."""
    return NP * NS * T + len(ROUTES) * T + p * NK * T + k * T + t
```

</details>

---

## Part 2: S3 Capacity Constraints (25 min)

### Context

Spot supplier S3 has limited capacity: **150,000 L per month** total across all plants.

In our model, S3 is supplier index `s=2`. The constraint for month `t` is:

```
x[0, 2, t] + x[1, 2, t] + x[2, 2, t] ≤ 150,000
```

We need 12 such constraints (one per month).

### Your Task

Build two arrays:
- **A_ub**: shape (12, 252) — coefficient matrix
- **b_ub**: shape (12,) — right-hand side (capacity limit)

The LP solver will enforce `A_ub @ x ≤ b_ub`.

**Hint:** Row `t` of `A_ub` should have 1.0 at positions `idx_x(p, 2, t)` for p=0,1,2. All other entries are 0.

In [ ]:
# YOUR CODE HERE
def build_inequality_constraints(s3_cap: float):
    """Build S3 monthly capacity cap: Σ_p x[p, 2, t] ≤ s3_cap."""
    raise NotImplementedError("Implement build_inequality_constraints() function.")

### Test Your Implementation

In [ ]:
# Build constraints
A_ub, b_ub = build_inequality_constraints(150_000)

# Shape checks
assert A_ub.shape == (12, 252), f"A_ub shape should be (12, 252), got {A_ub.shape}"
assert b_ub.shape == (12,), f"b_ub shape should be (12,), got {b_ub.shape}"
print("✅ Matrix dimensions correct")

# Right-hand side check
assert np.all(b_ub == 150_000), "All b_ub entries should equal s3_cap"
print("✅ Right-hand side correct")

# Coefficient check
for t in range(T):
    row = A_ub[t, :]
    assert np.sum(row) == 3.0, f"Row {t} should sum to 3.0 (one per plant)"
    
    for p in range(NP):
        expected_idx = idx_x(p, 2, t)  # s=2 is S3
        assert row[expected_idx] == 1.0, f"Row {t} should have 1.0 at index {expected_idx}"
    
    nonzero_indices = np.where(row != 0)[0]
    assert len(nonzero_indices) == 3, f"Row {t} should have exactly 3 non-zero entries"

print("✅ All constraints correctly encode S3 capacity limits")
print(f"\nSparsity: {np.sum(A_ub != 0)} non-zero entries out of {A_ub.size} total")
print(f"           ({100 * np.sum(A_ub != 0) / A_ub.size:.1f}% non-zero)")

### Visualize the Constraint Matrix

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))

sns.heatmap(A_ub, cmap="YlOrRd", cbar_kws={"label": "Coefficient"}, 
            linewidths=0, ax=ax, vmin=0, vmax=1)

# Add vertical lines to separate variable blocks
ax.axvline(x=108, color="blue", linewidth=2, linestyle="--", alpha=0.7, label="x | u boundary")
ax.axvline(x=180, color="green", linewidth=2, linestyle="--", alpha=0.7, label="u | e boundary")

# Annotate variable blocks
ax.text(54, -1.5, "x[p,s,t]\n(spot purchases)", ha="center", va="bottom", 
        fontsize=10, fontweight="bold", color="darkred")
ax.text(144, -1.5, "u[r,t]\n(transfers)", ha="center", va="bottom", 
        fontsize=10, fontweight="bold", color="darkblue")
ax.text(216, -1.5, "e[p,k,t]\n(excess sales)", ha="center", va="bottom", 
        fontsize=10, fontweight="bold", color="darkgreen")

ax.set_xlabel("Variable Index", fontsize=11, fontweight="bold")
ax.set_ylabel("Month", fontsize=11, fontweight="bold")
ax.set_title("S3 Capacity Constraint Matrix (12 constraints × 252 variables)",
            fontsize=12, fontweight="bold", pad=20)
ax.set_yticks(np.arange(12) + 0.5)
ax.set_yticklabels([m[:3] for m in MONTHS], rotation=0)
ax.set_xticks([0, 107, 108, 179, 180, 251])
ax.set_xticklabels(["0", "107", "108", "179", "180", "251"])
ax.legend(loc="upper right", framealpha=0.9)

plt.tight_layout()
plt.show()

print("📊 Matrix structure:")
print(f"   • Only {np.sum(A_ub != 0)}/3,024 entries are non-zero (1.2%)")
print("   • Each month constrains exactly 3 variables (one per plant)")
print("   • S3 purchases are in columns 2, 14, 26, ... (every 12th from col 2)")

<details>
<summary><b>💡 Solution - Part 2</b> (click to expand — try on your own first!)</summary>

```python
def build_inequality_constraints(s3_cap: float):
    """Build S3 monthly capacity cap: Σ_p x[p, 2, t] ≤ s3_cap."""
    N_VARS = 252
    A_ub = np.zeros((T, N_VARS))
    b_ub = np.full(T, s3_cap)
    
    for t in range(T):
        for p in range(NP):
            A_ub[t, idx_x(p, 2, t)] = 1.0  # s=2 is S3
    
    return A_ub, b_ub
```

</details>

---

## Part 3: Solve and Interpret Shadow Prices (20 min)

### Now Let's Solve the Full LP!

The other constraint matrix (`build_equality_constraints` for mass balance) is pre-implemented. We'll use your indexing functions to solve the complete problem.

In [ ]:
# Monkey-patch our implementations into the module
import milk_balancing.lp_model as lp_model

lp_model.idx_x = idx_x
lp_model.idx_u = idx_u
lp_model.idx_e = idx_e
lp_model.build_inequality_constraints = build_inequality_constraints

from milk_balancing import load_data, solve_scenario

# Solve baseline scenario
data = load_data()
result = solve_scenario(data)

if "error" in result:
    print("❌ Solver failed:")
    for err in result["error"]:
        print(f"  - {err}")
else:
    print("✅ LP solved successfully!\n")
    print(f"Total cost: €{result['total_cost']:,.0f}")
    print(f"  Farm:      €{result['farm_cost']:,.0f}")
    print(f"  Spot:      €{result['spot_cost']:,.0f}")
    print(f"  Transport: €{result['trans_cost']:,.0f}")
    print(f"  Sales rev: €{result['sales_rev']:,.0f}")

### Interpreting Shadow Prices

The **shadow price** of a constraint tells you how much the optimal cost would change if you relaxed that constraint by one unit.

For S3 capacity:
- Positive shadow price → S3 cap is **binding** (we're hitting the limit)
- Zero shadow price → S3 cap is **slack** (we're not using full capacity)

In [ ]:
duals_s3 = result["duals_s3"]

print("S3 Shadow Prices (€/L):")
print("Month     Shadow Price    Interpretation")
print("-" * 55)
for t, month in enumerate(MONTHS):
    sp = duals_s3[t]
    status = "🔴 BINDING" if sp > 0.001 else "  slack"
    print(f"{month:8s}  {sp:12.4f}    {status}")

# Show actual S3 usage
x_sol = result["x_sol"]  # Shape: (NP, NS, T)
s3_usage = x_sol[:, 2, :].sum(axis=0)  # Total S3 purchases per month (s=2)

print("\nS3 Usage vs Capacity:")
for t, month in enumerate(MONTHS):
    pct = 100 * s3_usage[t] / 150_000
    bar = "█" * int(pct / 5)
    print(f"{month:8s}  {s3_usage[t] / 1000:6.1f} kL / 150 kL  ({pct:5.1f}%)  {bar}")

### Experiment: Capacity Sensitivity Analysis

Let's test how total cost changes with different S3 capacity levels. Shadow prices predict local sensitivity, but let's visualize the full relationship.

In [ ]:
# Test multiple S3 capacity levels
capacity_levels = np.linspace(100_000, 200_000, 11)  # From 100kL to 200kL
costs = []
s3_volumes = []

for cap in capacity_levels:
    result_cap = solve_scenario(data, s3_cap=cap)
    if 'error' not in result_cap:
        costs.append(result_cap['total_cost'])
        s3_volumes.append(result_cap['x_sol'][:, 2, :].sum())  # Total S3 volume used
    else:
        costs.append(None)
        s3_volumes.append(None)

# Visualize
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# 1. Cost vs Capacity
valid_idx = [i for i, c in enumerate(costs) if c is not None]
valid_caps = [capacity_levels[i]/1000 for i in valid_idx]
valid_costs = [costs[i]/1000 for i in valid_idx]

ax1.plot(valid_caps, valid_costs, 'o-', linewidth=2, markersize=8, color='steelblue')
ax1.axvline(150, color='red', linestyle='--', alpha=0.7, linewidth=2, label='Baseline (150kL)')
ax1.axhline(result['total_cost']/1000, color='red', linestyle='--', alpha=0.5)
ax1.set_xlabel('S3 Monthly Capacity (kL)', fontsize=12, fontweight='bold')
ax1.set_ylabel('Total Cost (k€)', fontsize=12, fontweight='bold')
ax1.set_title('Cost Sensitivity to S3 Capacity', fontsize=14, fontweight='bold')
ax1.grid(True, alpha=0.3)
ax1.legend(fontsize=11)

# 2. S3 Usage vs Available Capacity
valid_volumes = [s3_volumes[i]/1000 for i in valid_idx]
ax2.plot(valid_caps, valid_caps, '--', color='gray', alpha=0.5, label='Max Available')
ax2.plot(valid_caps, valid_volumes, 'o-', linewidth=2, markersize=8, color='coral')
ax2.axvline(150, color='red', linestyle='--', alpha=0.7, linewidth=2, label='Baseline')
ax2.fill_between(valid_caps, valid_volumes, valid_caps, alpha=0.2, color='lightgray')
ax2.set_xlabel('S3 Monthly Capacity (kL)', fontsize=12, fontweight='bold')
ax2.set_ylabel('S3 Volume Used (kL)', fontsize=12, fontweight='bold')
ax2.set_title('S3 Utilization vs Capacity', fontsize=14, fontweight='bold')
ax2.grid(True, alpha=0.3)
ax2.legend(fontsize=11)

plt.tight_layout()
plt.show()

# Analysis
baseline_cost = result['total_cost']
cost_at_200 = costs[-1] if costs[-1] is not None else baseline_cost
cost_at_100 = costs[0] if costs[0] is not None else baseline_cost

print("📊 Capacity Sensitivity Insights:\n")
print(f"  • Increasing capacity 100→200kL saves: €{cost_at_100 - cost_at_200:,.0f}")
print(f"  • Current utilization rate: {s3_usage.sum() / (150_000 * 12) * 100:.1f}%")
print(f"  • At baseline, marginal value: {(costs[5] - costs[4])/(capacity_levels[5] - capacity_levels[4]) if costs[5] and costs[4] else 0:.2f} €/L")
print(f"\n💡 The curve shows {'diminishing returns' if cost_at_100 - costs[5] > costs[5] - cost_at_200 else 'increasing returns'} to capacity expansion!")

---

## Part 4: Price Sensitivity Analysis (30 min)

### Context

Business decisions require understanding how solutions respond to price changes. Let's test different pricing scenarios.

### Your Task

Implement a function to test multiple price scenarios and compare results.

In [ ]:
# YOUR CODE HERE
def test_price_scenarios(data):
    """
    Test 3 price scenarios:
    1. Baseline (current prices)
    2. S3 discount (20% cheaper)
    3. All spot prices up (10% more expensive)
    
    Returns:
        DataFrame with columns: ['scenario', 'total_cost', 'savings']
    """
    raise NotImplementedError("Implement test_price_scenarios() function.")

### Test and Visualize

In [ ]:
scenarios = test_price_scenarios(data)
print("Price Scenario Analysis:")
print(scenarios)

# Visualize
fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.bar(range(len(scenarios)), scenarios['total_cost']/1000, 
              color=['gray', 'skyblue', 'coral'], alpha=0.8)
ax.set_xticks(range(len(scenarios)))
ax.set_xticklabels(scenarios['scenario'])
ax.set_ylabel('Total Cost (k€)')
ax.set_title('Cost by Scenario', fontweight='bold', fontsize=14)
ax.grid(True, alpha=0.3, axis='y')

# Add savings labels
for i, (bar, cost, savings) in enumerate(zip(bars, scenarios['total_cost'], scenarios['savings'])):
    if savings != 0:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
                f'{savings/1000:+.0f}k', ha='center', fontweight='bold',
                color='green' if savings > 0 else 'red', fontsize=12)

plt.tight_layout()
plt.show()

print(f"\n✅ Scenarios tested! Best scenario saves €{scenarios['savings'].max()/1000:.0f}k")

<details>
<summary><b>💡 Solution - Part 4</b> (click to expand — try on your own first!)</summary>

```python
def test_price_scenarios(data):
    """Test price scenarios."""
    scenarios = []
    
    # 1. Baseline
    baseline = solve_scenario(data)
    scenarios.append({
        'scenario': 'Baseline',
        'total_cost': baseline['total_cost'],
        'savings': 0
    })
    
    # 2. S3 20% cheaper
    s3_cheap = solve_scenario(data, spot_mult=[1.0, 1.0, 0.8])
    if 'error' not in s3_cheap:
        scenarios.append({
            'scenario': 'S3 -20%',
            'total_cost': s3_cheap['total_cost'],
            'savings': baseline['total_cost'] - s3_cheap['total_cost']
        })
    
    # 3. All spot 10% more expensive
    all_expensive = solve_scenario(data, spot_mult=[1.1, 1.1, 1.1])
    if 'error' not in all_expensive:
        scenarios.append({
            'scenario': 'All Spot +10%',
            'total_cost': all_expensive['total_cost'],
            'savings': baseline['total_cost'] - all_expensive['total_cost']
        })
    
    return pd.DataFrame(scenarios)
```

</details>

---

## Part 5: Supplier Analysis (25 min)

### Context

Understanding supplier usage patterns helps identify sourcing dependencies and optimization opportunities.

### Your Task

Analyze how much volume comes from each supplier and visualize the cost breakdown.

In [ ]:
# YOUR CODE HERE
def analyze_supplier_usage(result, data):
    """
    Compute volume and average price by supplier.
    
    Returns:
        DataFrame with columns: ['supplier', 'volume_L', 'avg_price_per_L']
    """
    raise NotImplementedError("Implement analyze_supplier_usage() function.")

### Test and Visualize

In [ ]:
supplier_analysis = analyze_supplier_usage(result, data)
print("Supplier Analysis:")
print(supplier_analysis)

# Visualize
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# 1. Volume pie chart
ax1.pie(supplier_analysis['volume_L'], labels=supplier_analysis['supplier'], 
        autopct='%1.1f%%', startangle=90, colors=['#ff9999','#66b3ff','#99ff99'])
ax1.set_title('Supplier Volume Share', fontweight='bold', fontsize=14)

# 2. Cost breakdown bar chart
total_cost = result['total_cost']
cost_components = {
    'Farm': result['farm_cost'],
    'Spot': result['spot_cost'],
    'Transport': result['trans_cost']
}

ax2.bar(cost_components.keys(), 
        [v/1000 for v in cost_components.values()],
        color=['skyblue', 'coral', 'lightgreen'], alpha=0.8)
ax2.set_ylabel('Cost (k€)')
ax2.set_title('Total Cost Breakdown', fontweight='bold', fontsize=14)
ax2.grid(True, alpha=0.3, axis='y')

# Add percentage labels
for i, (component, cost) in enumerate(cost_components.items()):
    pct = cost / total_cost * 100
    ax2.text(i, cost/1000 + 10, f"{pct:.1f}%", ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

print(f"\n✅ Analysis complete! Spot purchases are {result['spot_cost']/total_cost*100:.1f}% of total cost")

<details>
<summary><b>💡 Solution - Part 5</b> (click to expand — try on your own first!)</summary>

```python
def analyze_supplier_usage(result, data):
    """Analyze supplier usage."""
    x_sol = result['x_sol']  # (NP, NS, T)
    c_eff = result['c_eff']  # (NP, NS, T) - effective cost including transport
    
    suppliers_data = []
    for s in range(NS):
        # Total volume purchased from this supplier across all plants and months
        volume = x_sol[:, s, :].sum()
        
        # Calculate weighted average price
        if volume > 0:
            # Sum of (volume * effective_price) / total volume
            total_cost = (x_sol[:, s, :] * c_eff[:, s, :]).sum()
            avg_price = total_cost / volume
        else:
            avg_price = 0
        
        suppliers_data.append({
            'supplier': f'S{s+1}',
            'volume_L': volume,
            'avg_price_per_L': avg_price
        })
    
    return pd.DataFrame(suppliers_data)

```

</details>

---

## 🎉 Workshop Complete!

### What You've Accomplished

1. **LP Modeling Fundamentals**
   - Variable indexing for 252-variable problems
   - Constraint matrix construction
   - Solving with scipy.linprog

2. **Economic Interpretation**
   - Shadow prices and dual values
   - Binding vs slack constraints
   - Marginal cost of capacity

3. **Sensitivity Analysis**
   - Price scenario testing
   - Impact quantification
   - What-if analysis

4. **Business Analytics**
   - Supplier usage patterns
   - Cost breakdown
   - Data visualization

### Key Insights from Your Analysis

Review your results to answer:
- Which supplier is most important by volume?
- Which months have binding S3 constraints?
- What scenario offers the best cost savings?
- What percentage of cost is farm vs spot vs transport?

### Real-World Applications

These techniques are used daily by:
- **Supply chain managers** at Danone, Nestlé, Unilever
- **Operations researchers** at Amazon, Walmart, FedEx
- **Management consultants** at McKinsey, BCG, Bain
- **Data scientists** in manufacturing, logistics, finance

### Next Steps

**Explore the full codebase:**
```bash
# Run the test suite
pytest tests/ -v

# Launch the interactive dashboard
streamlit run app.py
```

**Advanced topics:**
- Multi-objective optimization (cost vs risk trade-offs)
- Robust optimization (handling uncertainty)
- Integer programming (discrete decisions)
- Network flow algorithms

**Resources:**
- [scipy.optimize documentation](https://docs.scipy.org/doc/scipy/reference/optimize.html)
- [INFORMS (OR professional society)](https://www.informs.org/)
- [Linear Programming textbooks](https://www.wiley.com/Linear+Programming)

---

**Excellent work!** You're now equipped to tackle real-world optimization problems. 🚀